# Churn analysis: IBM Telco dataset
Public telecom data, not a SaaS company. Retention curves are by **tenure**, not calendar cohorts.

## 1. Load

In [ ]:
import sqlite3, pandas as pd, matplotlib.pyplot as plt
from churn_lib import CSV_PATH, DB_PATH, clean, load_customers, km_by_contract, fit_model
raw = pd.read_csv(CSV_PATH)
raw.shape

## 2. Clean
**Decision (edit if you disagree):** `TotalCharges` is blank for 11 customers, all with `tenure = 0` (new, never billed). I drop them: no history, and a zero duration cannot be used in Kaplan-Meier.

In [ ]:
blank = pd.to_numeric(raw["TotalCharges"], errors="coerce").isna()
print(blank.sum(), "blank TotalCharges; tenure values:", raw.loc[blank, "tenure"].unique())
df = clean(raw)
with sqlite3.connect(DB_PATH) as conn:
    df.to_sql("customers", conn, if_exists="replace", index=False)
print(df.shape, "churn rate:", round(df.Churn.mean(), 4))

## 3. Metrics
Run the queries in `metrics.sql` (one at a time in a SQLite client, or below).

In [ ]:
conn = sqlite3.connect(DB_PATH)
for q in open("metrics.sql").read().split(";"):
    if "SELECT" in q.upper():
        print(q.strip().splitlines()[0]); display(pd.read_sql(q, conn))

## 4. Retention curves (Kaplan-Meier)

In [ ]:
df = load_customers()
fits, table = km_by_contract(df)
fig, ax = plt.subplots()
for k in fits.values(): k.plot_survival_function(ax=ax)
ax.set_xlabel("Tenure (months)"); ax.set_ylabel("Share retained")
plt.savefig("screenshots/retention_curves.png", dpi=150, bbox_inches="tight")
table.style.format("{:.1%}")

## 5. Churn driver model
`TotalCharges` excluded (≈ tenure × MonthlyCharges). ~27% churn, so accuracy alone misleads: compare recall and AUC against the always-'no churn' baseline. Coefficients are *associated with* churn, not causes.

In [ ]:
model, metrics, odds = fit_model(df)
display(metrics.round(3)); display(odds.head(10)); display(odds.tail(10))

## 6. Notes for the write-up
- LTV assumes monthly churn ≈ overall churn ÷ average tenure.
- Numbers to copy into the memo: ...